# DeepFaceLive — Kaggle headless inference test

This notebook checks still-image face swapping on Kaggle **without opening the desktop app and without renting a paid cloud pod**.

## Before running
1. Open this notebook in Kaggle.
2. In the notebook's **Settings / Session options**, choose a **GPU** accelerator.
3. Make sure **Internet** is enabled.
4. Run the cells from top to bottom. The first download includes about 730 MB of model parts; allow it time to finish.

The first test uses the project's sample images of fictional demo people. It writes a face-crop result to `/kaggle/working/swapped_face.png`. The test does not yet measure live-video speed.

In [ ]:
# Check the actual GPU and driver. Stop immediately if no GPU was assigned.
import subprocess, sys, platform
subprocess.run(['nvidia-smi'], check=True)
print('Python:', sys.version)
print('Platform:', platform.platform())


In [ ]:
# Step 2 — Download only the source code and model files needed for this test.
# Model weights are large; sparse checkout skips unrelated video demos and extra models.
from pathlib import Path
import subprocess

REPO_DIR = Path('/kaggle/working/DeepFaceLive')
BRANCH_URL = 'https://github.com/johnlilda2-ui/DeepFaceLive.git'
BRANCH = 'experiment/cloud-headless-webrtc'

if not (REPO_DIR / '.git').exists():
    subprocess.run([
        'git', 'clone', '--filter=blob:none', '--depth', '1', '--sparse',
        '--single-branch', '--branch', BRANCH, BRANCH_URL, str(REPO_DIR)
    ], check=True)

# Include Python code plus only the 4 model weights this test uses.
patterns = [
    '/scripts/',
    '/xlib/',
    '/modelhub/__init__.py',
    '/modelhub/onnx/__init__.py',
    '/modelhub/onnx/*/*.py',
    '/modelhub/onnx/InsightFaceSwap/inswapper_128.onnx.part*',
    '/modelhub/onnx/InsightFaceSwap/w600k_r50.onnx.part*',
    '/modelhub/onnx/InsightFace2d106/InsightFace2D106.onnx',
    '/modelhub/onnx/YoloV5Face/YoloV5Face.onnx',
    '/doc/celebs/Rob_Doe/Rob_Doe.png',
    '/doc/celebs/Millie_Park/Millie_Park.png',
]
subprocess.run(['git', '-C', str(REPO_DIR), 'sparse-checkout', 'set', '--no-cone', *patterns], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '-B', BRANCH, 'FETCH_HEAD'], check=True)

required = [
    'modelhub/onnx/InsightFaceSwap/inswapper_128.onnx.part0',
    'modelhub/onnx/InsightFaceSwap/w600k_r50.onnx.part0',
    'modelhub/onnx/InsightFace2d106/InsightFace2D106.onnx',
    'modelhub/onnx/YoloV5Face/YoloV5Face.onnx',
    'doc/celebs/Rob_Doe/Rob_Doe.png',
    'doc/celebs/Millie_Park/Millie_Park.png',
]
missing = [p for p in required if not (REPO_DIR / p).is_file()]
if missing:
    raise FileNotFoundError('Files were not checked out: ' + ', '.join(missing))
print('Repository and required model files are ready:', REPO_DIR)
print('Testing commit:', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
# Use an isolated CUDA 12 runtime instead of ORT 1.31's CUDA 13 build.
# Do not uninstall or replace Kaggle's notebook packages.
import subprocess, sys
from pathlib import Path
ENV_DIR = Path('/kaggle/working/dfl-cuda12-env')
subprocess.run([sys.executable, '-m', 'venv', str(ENV_DIR)], check=True)
PYTHON = str(ENV_DIR / 'bin/python')
subprocess.run([
    PYTHON, '-m', 'pip', 'install', '--quiet', '--only-binary=:all:',
    'onnxruntime-gpu[cuda,cudnn]==1.22.0', 'numpy==2.2.6',
    'onnx==1.18.0', 'opencv-python-headless==4.11.0.86', 'numexpr==2.10.2', 'h5py==3.12.1'
], check=True)
verify_code = '''
import onnxruntime as ort
print('ONNX Runtime:', ort.__version__)
ort.preload_dlls(directory='')
ort.print_debug_info()
if 'CUDAExecutionProvider' not in ort.get_available_providers():
    raise RuntimeError('CUDA provider is unavailable')
print('Dependencies checked; execution is verified separately below.')
'''
subprocess.run([PYTHON, '-c', verify_code], check=True)
freeze = subprocess.run([PYTHON, '-m', 'pip', 'freeze'], check=True, capture_output=True, text=True)
Path('/kaggle/working/dfl-dependencies.txt').write_text(freeze.stdout)
print(freeze.stdout)


In [ ]:
# Step 4 — Run the smoke test using two supplied sample images.
# --require-cuda prevents a silent CPU fallback.
import subprocess, sys
from IPython.display import display, Image

source_image = REPO_DIR / 'doc/celebs/Rob_Doe/Rob_Doe.png'
target_image = REPO_DIR / 'doc/celebs/Millie_Park/Millie_Park.png'
output_image = Path('/kaggle/working/swapped_face.png')
command = [
    PYTHON, str(REPO_DIR / 'scripts/headless_inference_smoke_test.py'),
    '--source-image', str(source_image),
    '--target-image', str(target_image),
    '--output', str(output_image),
    '--require-cuda',
    '--evidence-dir', '/kaggle/working/dfl-gpu-evidence',
]
print('Running:', ' '.join(command))
# Stream logs to the cell and retain a downloadable copy.
log_path = Path('/kaggle/working/dfl-gpu-test.log')
with log_path.open('w') as log:
    process = subprocess.Popen(command, cwd=REPO_DIR, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
    code = process.wait()
from types import SimpleNamespace
result = SimpleNamespace(returncode=code)
if result.returncode != 0:
    raise RuntimeError('The headless smoke test failed. Copy the [FAIL] line and traceback above and send them to ChatGPT.')
if not output_image.is_file():
    raise FileNotFoundError('Test reported success but output image was not found: ' + str(output_image))
print('PASS — output image created:', output_image, 'bytes:', output_image.stat().st_size)
display(Image(filename=str(output_image)))

## Evidence required

A pass requires executed CUDA compute nodes in all four model profiles, plus successful image inference. A listed or active CUDA provider alone is insufficient. CPU shape operations are allowed. Keep `dfl-gpu-test.log`, `dfl-dependencies.txt`, and the latest run folder under `dfl-gpu-evidence` (including `execution_evidence.json` and raw profiles). Do not proceed to the cloud worker until this Kaggle run passes. Use only fictional samples or images supplied with consent.
